# 04 - Data Catalog, Data Dictionary, and Business Glossary

**Clarity Analytics Center (CAC) — Data Governance Layer**

This notebook adds a lightweight governance layer to the existing  Clarity Analytics Center SQLite prototype.

It implements three governance artifacts:

1. **Data Catalog** — dataset-level metadata for discoverability.
2. **Data Dictionary** — column-level technical and business metadata.
3. **Business Glossary** — consistent definitions for important CAC analytical terms.

### Design principle
The existing '**clarity_analytics_center.db**' is treated as the source of truth.  
This notebook connects to the database, discovers its implemented schema, and adds only new governance tables.

**Error Handling:** If the existing database file cannot be found, this notebook stops rather than creating a new empty database.

**Contributing members:**

---


Sonali Manohar


## 1. Connect to the existing Clarity Analytics Center SQLite database - clarity_analytics_center.db

This uses the same shared-drive location as the current storage/processing implementation.


In [ ]:
import sqlite3
import pandas as pd
from pathlib import Path

USE_DRIVE = False  # set True only if running against the team's shared Drive

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    db_path = Path(
        "/content/drive/Shareddrives/Group 4 Clarity Analytics "
        "Center/Clarity Analytics Center Database/clarity_analytics_center.db"
    )
else:
    NOTEBOOK_DIR = Path.cwd()
    REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
    db_path = REPO_ROOT / "data" / "clarity_analytics_center.db"

# Do NOT create a new database accidentally.
if not db_path.exists():
    raise FileNotFoundError(
        f"Existing CAC database was not found at: {db_path}\n"
        "Check that the database already exists (either locally in data/, or on the Group 4 Shared Drive if USE_DRIVE is True)."
    )

conn = sqlite3.connect(db_path)
conn.execute("PRAGMA foreign_keys = ON;")

print(f"Connected to existing CAC database: {db_path}")


Mounted at /content/drive
Connected to existing CAC database: /content/drive/Shareddrives/Group 4 Clarity Analytics Center/Clarity Analytics Center Database/clarity_analytics_center.db


## 2. Verify the currently implemented database

Before creating governance metadata, inspect the tables that actually exist.  
This keeps the catalog grounded in the prototype rather than an earlier design document.


In [2]:
existing_tables_df = pd.read_sql_query(
    '''
    SELECT name AS table_name
    FROM sqlite_master
    WHERE type = 'table'
      AND name NOT LIKE 'sqlite_%'
    ORDER BY name
    ''',
    conn
)

existing_tables_df


,table_name
0,bronze_ice_budget
1,bronze_ice_enforcement_metrics
2,bronze_ice_enforcement_pdfs
3,bronze_ice_operations
4,bronze_treasury_reconciliation
5,gold_dim_date
6,gold_dim_facility
7,gold_dim_org_unit
8,gold_dim_role
9,gold_dim_treasury_line


## 3. Create governance metadata tables

The governance tables are separate from the Medallion data tables and do not alter any existing records.

- **governance_data_catalog** stores dataset/table-level metadata.
- **governance_data_dictionary** stores column-level metadata.
- **governance_business_glossary** stores standardized business definitions.


In [3]:
conn.executescript(
    '''
    CREATE TABLE IF NOT EXISTS governance_data_catalog (
        table_name           TEXT PRIMARY KEY,
        layer                TEXT NOT NULL,
        domain               TEXT NOT NULL,
        description          TEXT NOT NULL,
        grain                TEXT,
        source_system        TEXT,
        refresh_frequency    TEXT,
        data_owner           TEXT,
        data_steward         TEXT,
        sensitivity_level    TEXT NOT NULL,
        contains_pii         INTEGER NOT NULL DEFAULT 0,
        primary_key          TEXT,
        known_limitations    TEXT,
        catalog_updated_at   TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP,
        CHECK (contains_pii IN (0, 1))
    );

    CREATE TABLE IF NOT EXISTS governance_data_dictionary (
        table_name           TEXT NOT NULL,
        column_name          TEXT NOT NULL,
        data_type            TEXT,
        is_nullable          INTEGER NOT NULL,
        is_primary_key       INTEGER NOT NULL,
        business_definition  TEXT,
        sensitivity_level    TEXT,
        source_or_derivation TEXT,
        dictionary_updated_at TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP,
        PRIMARY KEY (table_name, column_name),
        CHECK (is_nullable IN (0, 1)),
        CHECK (is_primary_key IN (0, 1))
    );

    CREATE TABLE IF NOT EXISTS governance_business_glossary (
        term                 TEXT PRIMARY KEY,
        definition           TEXT NOT NULL,
        domain               TEXT NOT NULL,
        authoritative_source TEXT,
        related_table        TEXT,
        related_column       TEXT,
        business_owner       TEXT,
        data_steward         TEXT,
        notes                TEXT,
        glossary_updated_at  TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP
    );
    '''
)

conn.commit()
print("Governance tables are ready.")


Governance tables are ready.


## 4. Populate the Data Catalog

The catalog describes the **implemented** Bronze, Silver, and Gold tables.  
Descriptions and grains are intentionally conservative: where the prototype does not establish a fact, the catalog does not claim one.

Ownership fields use CAC prototype roles rather than personal names so responsibility is documented without tying the model to a specific team member.


In [4]:
catalog_rows = [
    # ---------------- BRONZE ----------------
    {
        "table_name": "bronze_ice_operations",
        "layer": "Bronze",
        "domain": "Operations",
        "description": "Raw synthetic ICE staffing and assignment records landed for downstream processing.",
        "grain": "One source operations/assignment record.",
        "source_system": "Synthetic ICE operations dataset",
        "refresh_frequency": "Prototype batch load",
        "data_owner": "CAC Operations Data Owner",
        "data_steward": "CAC Operations Data Steward",
        "sensitivity_level": "Internal / Synthetic",
        "contains_pii": 0,
        "primary_key": None,
        "known_limitations": "Synthetic data approximates operational structures and is not actual person-level ICE staffing data."
    },
    {
        "table_name": "bronze_ice_budget",
        "layer": "Bronze",
        "domain": "Funding",
        "description": "Raw synthetic ICE budget records used for funding analysis.",
        "grain": "One source budget record for a fiscal year and department code.",
        "source_system": "Synthetic ICE budget dataset",
        "refresh_frequency": "Prototype batch load",
        "data_owner": "CAC Funding Data Owner",
        "data_steward": "CAC Funding Data Steward",
        "sensitivity_level": "Internal / Synthetic",
        "contains_pii": 0,
        "primary_key": None,
        "known_limitations": "ICE-level values are synthetic and should not be interpreted as official ICE financial figures."
    },
    {
        "table_name": "bronze_treasury_reconciliation",
        "layer": "Bronze",
        "domain": "Funding",
        "description": "Raw Treasury reconciliation records before Silver standardization.",
        "grain": "One source Treasury reconciliation line record.",
        "source_system": "U.S. Treasury Fiscal Data API",
        "refresh_frequency": "Prototype API batch load",
        "data_owner": "CAC Funding Data Owner",
        "data_steward": "CAC Funding Data Steward",
        "sensitivity_level": "Public Source",
        "contains_pii": 0,
        "primary_key": None,
        "known_limitations": "Source labels, types, and restatement fields require standardization before analytical use."
    },
    {
        "table_name": "bronze_ice_enforcement_pdfs",
        "layer": "Bronze",
        "domain": "Enforcement",
        "description": "Page-level text extracted from ICE annual report PDFs with extraction metadata.",
        "grain": "One extracted PDF page.",
        "source_system": "ICE Annual Reports",
        "refresh_frequency": "Prototype annual-report batch load",
        "data_owner": "CAC Enforcement Data Owner",
        "data_steward": "CAC Enforcement Data Steward",
        "sensitivity_level": "Public Source",
        "contains_pii": 0,
        "primary_key": None,
        "known_limitations": "PDF layout and extraction quality may affect text completeness."
    },
    {
        "table_name": "bronze_ice_enforcement_metrics",
        "layer": "Bronze",
        "domain": "Enforcement",
        "description": "Structured enforcement metrics derived from ICE annual-report content.",
        "grain": "One extracted metric associated with a report/page.",
        "source_system": "ICE Annual Reports / metric extraction process",
        "refresh_frequency": "Prototype annual-report batch load",
        "data_owner": "CAC Enforcement Data Owner",
        "data_steward": "CAC Enforcement Data Steward",
        "sensitivity_level": "Public Source",
        "contains_pii": 0,
        "primary_key": None,
        "known_limitations": "Metrics must retain source-page and metric-comment context because reported measures may have source-specific caveats."
    },

    # ---------------- SILVER ----------------
    {
        "table_name": "silver_operations",
        "layer": "Silver",
        "domain": "Operations",
        "description": "Cleaned and standardized operations records with active-assignment status.",
        "grain": "One processed operations/assignment record.",
        "source_system": "bronze_ice_operations",
        "refresh_frequency": "Prototype batch processing",
        "data_owner": "CAC Operations Data Owner",
        "data_steward": "CAC Operations Data Steward",
        "sensitivity_level": "Internal / Synthetic",
        "contains_pii": 0,
        "primary_key": "operations_row_id",
        "known_limitations": "Underlying staffing records are synthetic."
    },
    {
        "table_name": "silver_budget",
        "layer": "Silver",
        "domain": "Funding",
        "description": "Cleaned ICE budget records with deterministic row identifiers and processing timestamp.",
        "grain": "One processed budget record.",
        "source_system": "bronze_ice_budget",
        "refresh_frequency": "Prototype batch processing",
        "data_owner": "CAC Funding Data Owner",
        "data_steward": "CAC Funding Data Steward",
        "sensitivity_level": "Internal / Synthetic",
        "contains_pii": 0,
        "primary_key": "budget_row_id",
        "known_limitations": "ICE-level values are synthetic and are not an external accuracy reconciliation to Treasury totals."
    },
    {
        "table_name": "silver_treasury_reconciliation",
        "layer": "Silver",
        "domain": "Funding",
        "description": "Standardized Treasury reconciliation data with cleaned labels, typed values, deterministic keys, and processing timestamp.",
        "grain": "One unique record_date, source_line_number, and restatement_flag combination.",
        "source_system": "bronze_treasury_reconciliation",
        "refresh_frequency": "Prototype batch processing",
        "data_owner": "CAC Funding Data Owner",
        "data_steward": "CAC Funding Data Steward",
        "sensitivity_level": "Public Source",
        "contains_pii": 0,
        "primary_key": "treasury_reconciliation_row_id",
        "known_limitations": "Financial interpretation must distinguish statement fiscal year from record/publication fiscal year."
    },
    {
        "table_name": "silver_enforcement",
        "layer": "Silver",
        "domain": "Enforcement",
        "description": "Standardized enforcement records linking extracted metrics to report-page provenance.",
        "grain": "One processed enforcement metric/report-page record.",
        "source_system": "bronze_ice_enforcement_pdfs + bronze_ice_enforcement_metrics",
        "refresh_frequency": "Prototype batch processing",
        "data_owner": "CAC Enforcement Data Owner",
        "data_steward": "CAC Enforcement Data Steward",
        "sensitivity_level": "Public Source",
        "contains_pii": 0,
        "primary_key": "enforcement_row_id",
        "known_limitations": "Metric definitions and caveats may vary by annual report and should be interpreted with metric_comment and source page."
    },

    # ---------------- GOLD DIMENSIONS ----------------
    {
        "table_name": "gold_dim_date",
        "layer": "Gold",
        "domain": "Shared",
        "description": "Shared calendar dimension used to align staffing, funding, Treasury, and enforcement facts.",
        "grain": "One calendar date.",
        "source_system": "Derived in processing layer",
        "refresh_frequency": "Prototype build",
        "data_owner": "CAC Analytics Data Owner",
        "data_steward": "CAC Analytics Data Steward",
        "sensitivity_level": "Internal Analytical",
        "contains_pii": 0,
        "primary_key": "date_key",
        "known_limitations": None
    },
    {
        "table_name": "gold_dim_org_unit",
        "layer": "Gold",
        "domain": "Operations",
        "description": "Stable organization-unit dimension for department and unit combinations.",
        "grain": "One unique department and unit combination.",
        "source_system": "silver_operations",
        "refresh_frequency": "Prototype batch processing",
        "data_owner": "CAC Operations Data Owner",
        "data_steward": "CAC Operations Data Steward",
        "sensitivity_level": "Internal / Synthetic",
        "contains_pii": 0,
        "primary_key": "org_unit_key",
        "known_limitations": "Organization structures are derived from synthetic staffing data."
    },
    {
        "table_name": "gold_dim_role",
        "layer": "Gold",
        "domain": "Operations",
        "description": "Standardized role dimension including seniority and leadership classification.",
        "grain": "One role.",
        "source_system": "silver_operations / processing mapping",
        "refresh_frequency": "Prototype batch processing",
        "data_owner": "CAC Operations Data Owner",
        "data_steward": "CAC Operations Data Steward",
        "sensitivity_level": "Internal / Synthetic",
        "contains_pii": 0,
        "primary_key": "role_key",
        "known_limitations": "Role hierarchy reflects the prototype's predefined role mapping."
    },
    {
        "table_name": "gold_dim_facility",
        "layer": "Gold",
        "domain": "Operations",
        "description": "Facility dimension separating site name, state code, facility type, and display label.",
        "grain": "One facility.",
        "source_system": "silver_operations",
        "refresh_frequency": "Prototype batch processing",
        "data_owner": "CAC Operations Data Owner",
        "data_steward": "CAC Operations Data Steward",
        "sensitivity_level": "Internal / Synthetic",
        "contains_pii": 0,
        "primary_key": "facility_key",
        "known_limitations": "Facility attributes are derived from the synthetic operations dataset and require the expected 'Name (ST)' source format."
    },
    {
        "table_name": "gold_dim_treasury_line",
        "layer": "Gold",
        "domain": "Funding",
        "description": "Treasury financial-statement line dimension used to classify line items for analysis.",
        "grain": "One unique Treasury component and line-item combination.",
        "source_system": "silver_treasury_reconciliation",
        "refresh_frequency": "Prototype batch processing",
        "data_owner": "CAC Funding Data Owner",
        "data_steward": "CAC Funding Data Steward",
        "sensitivity_level": "Public Source",
        "contains_pii": 0,
        "primary_key": "treasury_line_key",
        "known_limitations": "Only rows classified as additive detail should be summed without risk of double counting subtotals/totals."
    },

    # ---------------- GOLD FACTS ----------------
    {
        "table_name": "gold_fact_budget",
        "layer": "Gold",
        "domain": "Funding",
        "description": "Analytical budget fact including standardized fiscal-year date key and accrual-to-cash gap.",
        "grain": "One Silver budget row represented as a Gold budget fact.",
        "source_system": "silver_budget",
        "refresh_frequency": "Prototype batch processing",
        "data_owner": "CAC Funding Data Owner",
        "data_steward": "CAC Funding Data Steward",
        "sensitivity_level": "Internal / Synthetic",
        "contains_pii": 0,
        "primary_key": "budget_key",
        "known_limitations": "ICE budget amounts are synthetic; the table does not by itself prove reconciliation to official Treasury totals."
    },
    {
        "table_name": "gold_fact_treasury",
        "layer": "Gold",
        "domain": "Funding",
        "description": "Treasury statement fact preserving original/restated status and the distinction between statement and record fiscal years.",
        "grain": "One statement fiscal year × Treasury line item × restatement occurrence as represented by the source record.",
        "source_system": "silver_treasury_reconciliation",
        "refresh_frequency": "Prototype batch processing",
        "data_owner": "CAC Funding Data Owner",
        "data_steward": "CAC Funding Data Steward",
        "sensitivity_level": "Public Source",
        "contains_pii": 0,
        "primary_key": "treasury_key",
        "known_limitations": "Analysts must not confuse the fiscal year described by the statement with the fiscal year in which the record was published."
    },
    {
        "table_name": "gold_fact_enforcement",
        "layer": "Gold",
        "domain": "Enforcement",
        "description": "Curated enforcement metrics with source caveats and report-page provenance.",
        "grain": "One fiscal year and metric_name pair.",
        "source_system": "silver_enforcement",
        "refresh_frequency": "Prototype batch processing",
        "data_owner": "CAC Enforcement Data Owner",
        "data_steward": "CAC Enforcement Data Steward",
        "sensitivity_level": "Public Source",
        "contains_pii": 0,
        "primary_key": "enforcement_key",
        "known_limitations": "Reported metrics may have source-specific definitions or caveats; use metric_comment and source_page."
    },
    {
        "table_name": "gold_fact_assignment",
        "layer": "Gold",
        "domain": "Operations",
        "description": "Gold assignment fact defined by the implemented schema for staffing analysis.",
        "grain": "One assignment/posting record.",
        "source_system": "silver_operations",
        "refresh_frequency": "Prototype batch processing",
        "data_owner": "CAC Operations Data Owner",
        "data_steward": "CAC Operations Data Steward",
        "sensitivity_level": "Internal / Synthetic",
        "contains_pii": 0,
        "primary_key": "assignment_key",
        "known_limitations": "The table exists in the implemented schema; verify the current processing run has fully populated it before relying on it analytically."
    }
]

catalog_df = pd.DataFrame(catalog_rows)

# Only catalog data tables that actually exist in the current database.
actual_table_names = set(
    pd.read_sql_query(
        "SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%'",
        conn
    )["name"]
)

catalog_df = catalog_df[catalog_df["table_name"].isin(actual_table_names)].copy()

catalog_upsert_sql = '''
INSERT INTO governance_data_catalog (
    table_name, layer, domain, description, grain, source_system,
    refresh_frequency, data_owner, data_steward, sensitivity_level,
    contains_pii, primary_key, known_limitations, catalog_updated_at
)
VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, CURRENT_TIMESTAMP)
ON CONFLICT(table_name) DO UPDATE SET
    layer = excluded.layer,
    domain = excluded.domain,
    description = excluded.description,
    grain = excluded.grain,
    source_system = excluded.source_system,
    refresh_frequency = excluded.refresh_frequency,
    data_owner = excluded.data_owner,
    data_steward = excluded.data_steward,
    sensitivity_level = excluded.sensitivity_level,
    contains_pii = excluded.contains_pii,
    primary_key = excluded.primary_key,
    known_limitations = excluded.known_limitations,
    catalog_updated_at = CURRENT_TIMESTAMP;
'''

conn.executemany(
    catalog_upsert_sql,
    catalog_df[
        [
            "table_name", "layer", "domain", "description", "grain",
            "source_system", "refresh_frequency", "data_owner", "data_steward",
            "sensitivity_level", "contains_pii", "primary_key", "known_limitations"
        ]
    ].itertuples(index=False, name=None)
)
conn.commit()

print(f"Cataloged {len(catalog_df)} implemented data tables.")


Cataloged 18 implemented data tables.


In [ ]:
import pandas as pd

df = pd.read_sql_query(
    "SELECT * FROM governance_data_catalog",
    conn
)

display(df)

## 5. Build the Data Dictionary from the live SQLite schema

The technical portion of the dictionary is generated directly from SQLite `PRAGMA table_info`, so column names, data types, nullability, and primary-key status reflect the database that actually exists.

Selected business definitions and derivations are then added for important analytical fields.


In [6]:
# Business definitions for important fields.
# These are limited to meanings supported by the implemented schema and project documentation.
column_metadata = {
    ("silver_operations", "is_active_assignment"): (
        "Indicator showing whether the assignment is currently open in the processed operations data.",
        "Derived during Silver processing from assignment end-date status."
    ),
    ("silver_treasury_reconciliation", "statement_fiscal_year"): (
        "Fiscal year described by the Treasury financial statement.",
        "Standardized from the Bronze Treasury source."
    ),
    ("silver_treasury_reconciliation", "record_fiscal_year"): (
        "Fiscal year associated with the publication/record date rather than the year the statement describes.",
        "Standardized from the Bronze Treasury source."
    ),
    ("silver_treasury_reconciliation", "restatement_flag"): (
        "Source indicator identifying whether the financial statement value is a restatement.",
        "Standardized from the Treasury source."
    ),
    ("gold_fact_budget", "accrual_to_cash_gap"): (
        "Difference between net operating cost and budget deficit contribution in the prototype Gold model.",
        "Derived as net_operating_cost - budget_deficit_contribution."
    ),
    ("gold_fact_treasury", "is_restated"): (
        "Binary indicator identifying a restated Treasury record.",
        "Derived from restatement_flag during Gold processing."
    ),
    ("gold_fact_enforcement", "metric_comment"): (
        "Source-specific caveat or explanatory context attached to an enforcement metric.",
        "Carried forward from enforcement metric extraction."
    ),
    ("gold_fact_enforcement", "source_page"): (
        "Page of the ICE annual report associated with the reported enforcement metric.",
        "Carried forward from Silver enforcement provenance."
    ),
    ("gold_dim_treasury_line", "is_additive"): (
        "Indicator identifying detail rows that can be summed without intentionally including subtotal/total rows.",
        "Derived from the Treasury line-level classification."
    ),
    ("gold_dim_role", "is_leadership"): (
        "Indicator identifying roles classified as leadership in the prototype role mapping.",
        "Derived from the predefined Gold role mapping."
    ),
    ("gold_dim_date", "fiscal_year"): (
        "U.S. federal fiscal year associated with the calendar date.",
        "Derived in the shared Gold date dimension; federal fiscal year starts October 1."
    ),
    ("gold_dim_date", "fiscal_quarter"): (
        "Quarter within the U.S. federal fiscal year.",
        "Derived in the shared Gold date dimension."
    ),
}

# Sensitivity follows the table-level catalog classification unless overridden later.
catalog_sensitivity = dict(
    pd.read_sql_query(
        "SELECT table_name, sensitivity_level FROM governance_data_catalog", conn
    )[["table_name", "sensitivity_level"]].itertuples(index=False, name=None)
)

dictionary_rows = []

cataloged_tables = pd.read_sql_query(
    "SELECT table_name FROM governance_data_catalog ORDER BY table_name", conn
)["table_name"].tolist()

for table_name in cataloged_tables:
    schema_df = pd.read_sql_query(f"PRAGMA table_info('{table_name}')", conn)

    for _, row in schema_df.iterrows():
        key = (table_name, row["name"])
        business_definition, source_or_derivation = column_metadata.get(
            key, (None, None)
        )

        dictionary_rows.append({
            "table_name": table_name,
            "column_name": row["name"],
            "data_type": row["type"],
            # PRAGMA notnull=1 means NOT NULL. PK columns are also treated as non-nullable.
            "is_nullable": 0 if (row["notnull"] == 1 or row["pk"] == 1) else 1,
            "is_primary_key": 1 if row["pk"] else 0,
            "business_definition": business_definition,
            "sensitivity_level": catalog_sensitivity.get(table_name),
            "source_or_derivation": source_or_derivation
        })

dictionary_df = pd.DataFrame(dictionary_rows)

dictionary_upsert_sql = '''
INSERT INTO governance_data_dictionary (
    table_name, column_name, data_type, is_nullable, is_primary_key,
    business_definition, sensitivity_level, source_or_derivation,
    dictionary_updated_at
)
VALUES (?, ?, ?, ?, ?, ?, ?, ?, CURRENT_TIMESTAMP)
ON CONFLICT(table_name, column_name) DO UPDATE SET
    data_type = excluded.data_type,
    is_nullable = excluded.is_nullable,
    is_primary_key = excluded.is_primary_key,
    business_definition = COALESCE(excluded.business_definition, governance_data_dictionary.business_definition),
    sensitivity_level = excluded.sensitivity_level,
    source_or_derivation = COALESCE(excluded.source_or_derivation, governance_data_dictionary.source_or_derivation),
    dictionary_updated_at = CURRENT_TIMESTAMP;
'''

conn.executemany(
    dictionary_upsert_sql,
    dictionary_df[
        [
            "table_name", "column_name", "data_type", "is_nullable",
            "is_primary_key", "business_definition", "sensitivity_level",
            "source_or_derivation"
        ]
    ].itertuples(index=False, name=None)
)
conn.commit()

print(f"Documented {len(dictionary_df)} implemented columns.")


Documented 166 implemented columns.


In [7]:
import pandas as pd

df = pd.read_sql_query(
    "SELECT * FROM governance_data_dictionary",
    conn
)

display(df)

,table_name,column_name,data_type,is_nullable,is_primary_key,business_definition,sensitivity_level,source_or_derivation,dictionary_updated_at
0,bronze_ice_budget,fiscal_year,INTEGER,0,0,None,Internal / Synthetic,None,2026-08-13 19:42:50
1,bronze_ice_budget,department_code,TEXT,0,0,None,Internal / Synthetic,None,2026-08-13 19:42:50
2,bronze_ice_budget,agency_name,TEXT,0,0,None,Internal / Synthetic,None,2026-08-13 19:42:50
3,bronze_ice_budget,net_operating_cost,REAL,1,0,None,Internal / Synthetic,None,2026-08-13 19:42:50
4,bronze_ice_budget,budget_authority,REAL,1,0,None,Internal / Synthetic,None,2026-08-13 19:42:50
...,...,...,...,...,...,...,...,...,...
161,silver_treasury_reconciliation,record_calendar_year,INTEGER,1,0,None,Public Source,None,2026-08-13 19:42:50
162,silver_treasury_reconciliation,record_calendar_quarter,INTEGER,1,0,None,Public Source,None,2026-08-13 19:42:50
163,silver_treasury_reconciliation,record_calendar_month,INTEGER,1,0,None,Public Source,None,2026-08-13 19:42:50
164,silver_treasury_reconciliation,record_calendar_day,INTEGER,1,0,None,Public Source,None,2026-08-13 19:42:50


## 6. Populate the Business Glossary

The glossary standardizes terms that analysts may otherwise interpret differently across operations, funding, and enforcement analysis.

Definitions describe how the term is used **in this CAC prototype**. They are not presented as universal legal or agency definitions.


In [8]:
glossary_rows = [
    {
        "term": "Fiscal Year",
        "definition": "The U.S. federal fiscal year used to align CAC analytical data; it begins October 1 and ends September 30.",
        "domain": "Shared",
        "authoritative_source": "CAC Gold date dimension",
        "related_table": "gold_dim_date",
        "related_column": "fiscal_year",
        "business_owner": "CAC Analytics Data Owner",
        "data_steward": "CAC Analytics Data Steward",
        "notes": "Used as the common analytical time basis across funding and enforcement facts."
    },
    {
        "term": "Active Assignment",
        "definition": "An operations assignment treated by the prototype as currently open rather than completed.",
        "domain": "Operations",
        "authoritative_source": "CAC Silver operations processing",
        "related_table": "silver_operations",
        "related_column": "is_active_assignment",
        "business_owner": "CAC Operations Data Owner",
        "data_steward": "CAC Operations Data Steward",
        "notes": None
    },
    {
        "term": "Organization Unit",
        "definition": "A standardized department-and-unit combination used to slice staffing assignments in the Gold model.",
        "domain": "Operations",
        "authoritative_source": "CAC Gold model",
        "related_table": "gold_dim_org_unit",
        "related_column": "org_unit_key",
        "business_owner": "CAC Operations Data Owner",
        "data_steward": "CAC Operations Data Steward",
        "notes": None
    },
    {
        "term": "Leadership Role",
        "definition": "A job role classified as leadership by the prototype's predefined Gold role mapping.",
        "domain": "Operations",
        "authoritative_source": "CAC Gold role dimension",
        "related_table": "gold_dim_role",
        "related_column": "is_leadership",
        "business_owner": "CAC Operations Data Owner",
        "data_steward": "CAC Operations Data Steward",
        "notes": "The classification is a prototype analytical rule."
    },
    {
        "term": "Net Operating Cost",
        "definition": "The net operating cost amount carried into CAC funding analysis from the budget/Treasury-oriented data model.",
        "domain": "Funding",
        "authoritative_source": "CAC implemented funding schema",
        "related_table": "gold_fact_budget",
        "related_column": "net_operating_cost",
        "business_owner": "CAC Funding Data Owner",
        "data_steward": "CAC Funding Data Steward",
        "notes": "ICE-level budget values in the prototype are synthetic."
    },
    {
        "term": "Budget Authority",
        "definition": "The budget authority amount stored for a CAC budget fact and used alongside other funding measures.",
        "domain": "Funding",
        "authoritative_source": "CAC implemented funding schema",
        "related_table": "gold_fact_budget",
        "related_column": "budget_authority",
        "business_owner": "CAC Funding Data Owner",
        "data_steward": "CAC Funding Data Steward",
        "notes": "ICE-level budget values in the prototype are synthetic."
    },
    {
        "term": "Budget Deficit Contribution",
        "definition": "The budget-deficit contribution amount stored in the CAC budget fact and used in the prototype's accrual-to-cash gap calculation.",
        "domain": "Funding",
        "authoritative_source": "CAC implemented funding schema",
        "related_table": "gold_fact_budget",
        "related_column": "budget_deficit_contribution",
        "business_owner": "CAC Funding Data Owner",
        "data_steward": "CAC Funding Data Steward",
        "notes": "ICE-level budget values in the prototype are synthetic."
    },
    {
        "term": "Accrual-to-Cash Gap",
        "definition": "A Gold analytical measure calculated as net operating cost minus budget deficit contribution so the same formula is reused consistently.",
        "domain": "Funding",
        "authoritative_source": "CAC Gold processing logic",
        "related_table": "gold_fact_budget",
        "related_column": "accrual_to_cash_gap",
        "business_owner": "CAC Funding Data Owner",
        "data_steward": "CAC Funding Data Steward",
        "notes": "This is a CAC prototype analytical measure."
    },
    {
        "term": "Statement Fiscal Year",
        "definition": "The fiscal year that a Treasury statement value describes.",
        "domain": "Funding",
        "authoritative_source": "CAC Treasury model",
        "related_table": "gold_fact_treasury",
        "related_column": "statement_fiscal_year",
        "business_owner": "CAC Funding Data Owner",
        "data_steward": "CAC Funding Data Steward",
        "notes": "Do not confuse with record_fiscal_year."
    },
    {
        "term": "Record Fiscal Year",
        "definition": "The fiscal year associated with when a Treasury record was published/recorded, rather than the fiscal year the financial statement describes.",
        "domain": "Funding",
        "authoritative_source": "CAC Treasury model",
        "related_table": "gold_fact_treasury",
        "related_column": "record_fiscal_year",
        "business_owner": "CAC Funding Data Owner",
        "data_steward": "CAC Funding Data Steward",
        "notes": "Kept separately from statement_fiscal_year to support correct restatement analysis."
    },
    {
        "term": "Restatement",
        "definition": "A later version of a Treasury financial statement value identified by the source restatement indicator.",
        "domain": "Funding",
        "authoritative_source": "Treasury source as represented in CAC",
        "related_table": "gold_fact_treasury",
        "related_column": "is_restated",
        "business_owner": "CAC Funding Data Owner",
        "data_steward": "CAC Funding Data Steward",
        "notes": None
    },
    {
        "term": "Additive Treasury Line",
        "definition": "A Treasury line classified as a detail row that may be summed without intentionally adding subtotal or total rows to the same aggregation.",
        "domain": "Funding",
        "authoritative_source": "CAC Gold Treasury-line classification",
        "related_table": "gold_dim_treasury_line",
        "related_column": "is_additive",
        "business_owner": "CAC Funding Data Owner",
        "data_steward": "CAC Funding Data Steward",
        "notes": "Filtering to additive detail rows helps prevent double counting."
    },
    {
        "term": "Enforcement Metric",
        "definition": "A reported enforcement measure, such as an arrests, removals, or detentions figure, stored with its source context in the CAC Gold model.",
        "domain": "Enforcement",
        "authoritative_source": "ICE Annual Reports as represented in CAC",
        "related_table": "gold_fact_enforcement",
        "related_column": "metric_name",
        "business_owner": "CAC Enforcement Data Owner",
        "data_steward": "CAC Enforcement Data Steward",
        "notes": "Metric definitions may vary by source/report; consult metric_comment and source_page."
    },
    {
        "term": "Metric Comment",
        "definition": "Source-specific caveat or explanatory context retained with an enforcement metric to support correct interpretation.",
        "domain": "Enforcement",
        "authoritative_source": "ICE Annual Reports / CAC metric extraction",
        "related_table": "gold_fact_enforcement",
        "related_column": "metric_comment",
        "business_owner": "CAC Enforcement Data Owner",
        "data_steward": "CAC Enforcement Data Steward",
        "notes": None
    }
]

glossary_df = pd.DataFrame(glossary_rows)

glossary_upsert_sql = '''
INSERT INTO governance_business_glossary (
    term, definition, domain, authoritative_source, related_table,
    related_column, business_owner, data_steward, notes, glossary_updated_at
)
VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, CURRENT_TIMESTAMP)
ON CONFLICT(term) DO UPDATE SET
    definition = excluded.definition,
    domain = excluded.domain,
    authoritative_source = excluded.authoritative_source,
    related_table = excluded.related_table,
    related_column = excluded.related_column,
    business_owner = excluded.business_owner,
    data_steward = excluded.data_steward,
    notes = excluded.notes,
    glossary_updated_at = CURRENT_TIMESTAMP;
'''

conn.executemany(
    glossary_upsert_sql,
    glossary_df[
        [
            "term", "definition", "domain", "authoritative_source",
            "related_table", "related_column", "business_owner",
            "data_steward", "notes"
        ]
    ].itertuples(index=False, name=None)
)
conn.commit()

print(f"Loaded {len(glossary_df)} business glossary terms.")


Loaded 14 business glossary terms.


In [9]:
import pandas as pd

df = pd.read_sql_query(
    "SELECT * FROM governance_business_glossary",
    conn
)

display(df)

,term,definition,domain,authoritative_source,related_table,related_column,business_owner,data_steward,notes,glossary_updated_at
0,Fiscal Year,The U.S. federal fiscal year used to align CAC...,Shared,CAC Gold date dimension,gold_dim_date,fiscal_year,CAC Analytics Data Owner,CAC Analytics Data Steward,Used as the common analytical time basis acros...,2026-08-13 19:45:14
1,Active Assignment,An operations assignment treated by the protot...,Operations,CAC Silver operations processing,silver_operations,is_active_assignment,CAC Operations Data Owner,CAC Operations Data Steward,None,2026-08-13 19:45:14
2,Organization Unit,A standardized department-and-unit combination...,Operations,CAC Gold model,gold_dim_org_unit,org_unit_key,CAC Operations Data Owner,CAC Operations Data Steward,None,2026-08-13 19:45:14
3,Leadership Role,A job role classified as leadership by the pro...,Operations,CAC Gold role dimension,gold_dim_role,is_leadership,CAC Operations Data Owner,CAC Operations Data Steward,The classification is a prototype analytical r...,2026-08-13 19:45:14
4,Net Operating Cost,The net operating cost amount carried into CAC...,Funding,CAC implemented funding schema,gold_fact_budget,net_operating_cost,CAC Funding Data Owner,CAC Funding Data Steward,ICE-level budget values in the prototype are s...,2026-08-13 19:45:14
5,Budget Authority,The budget authority amount stored for a CAC b...,Funding,CAC implemented funding schema,gold_fact_budget,budget_authority,CAC Funding Data Owner,CAC Funding Data Steward,ICE-level budget values in the prototype are s...,2026-08-13 19:45:14
6,Budget Deficit Contribution,The budget-deficit contribution amount stored ...,Funding,CAC implemented funding schema,gold_fact_budget,budget_deficit_contribution,CAC Funding Data Owner,CAC Funding Data Steward,ICE-level budget values in the prototype are s...,2026-08-13 19:45:14
7,Accrual-to-Cash Gap,A Gold analytical measure calculated as net op...,Funding,CAC Gold processing logic,gold_fact_budget,accrual_to_cash_gap,CAC Funding Data Owner,CAC Funding Data Steward,This is a CAC prototype analytical measure.,2026-08-13 19:45:14
8,Statement Fiscal Year,The fiscal year that a Treasury statement valu...,Funding,CAC Treasury model,gold_fact_treasury,statement_fiscal_year,CAC Funding Data Owner,CAC Funding Data Steward,Do not confuse with record_fiscal_year.,2026-08-13 19:45:14
9,Record Fiscal Year,The fiscal year associated with when a Treasur...,Funding,CAC Treasury model,gold_fact_treasury,record_fiscal_year,CAC Funding Data Owner,CAC Funding Data Steward,Kept separately from statement_fiscal_year to ...,2026-08-13 19:45:14


## 7. Demonstrate data discoverability

Example: An analyst querys the catalog to identify available datasets so as to not rely only on informal team knowledge.


In [11]:
# Example 1: discover available Gold datasets
pd.read_sql_query(
    '''
    SELECT
        table_name,
        domain,
        description,
        grain,
        sensitivity_level,
        known_limitations
    FROM governance_data_catalog
    WHERE layer = 'Gold'
    ORDER BY domain, table_name
    ''',
    conn
)


,table_name,domain,description,grain,sensitivity_level,known_limitations
0,gold_fact_enforcement,Enforcement,Curated enforcement metrics with source caveat...,One fiscal year and metric_name pair.,Public Source,Reported metrics may have source-specific defi...
1,gold_dim_treasury_line,Funding,Treasury financial-statement line dimension us...,One unique Treasury component and line-item co...,Public Source,Only rows classified as additive detail should...
2,gold_fact_budget,Funding,Analytical budget fact including standardized ...,One Silver budget row represented as a Gold bu...,Internal / Synthetic,ICE budget amounts are synthetic; the table do...
3,gold_fact_treasury,Funding,Treasury statement fact preserving original/re...,One statement fiscal year × Treasury line item...,Public Source,Analysts must not confuse the fiscal year desc...
4,gold_dim_facility,Operations,"Facility dimension separating site name, state...",One facility.,Internal / Synthetic,Facility attributes are derived from the synth...
5,gold_dim_org_unit,Operations,Stable organization-unit dimension for departm...,One unique department and unit combination.,Internal / Synthetic,Organization structures are derived from synth...
6,gold_dim_role,Operations,Standardized role dimension including seniorit...,One role.,Internal / Synthetic,Role hierarchy reflects the prototype's predef...
7,gold_fact_assignment,Operations,Gold assignment fact defined by the implemente...,One assignment/posting record.,Internal / Synthetic,The table exists in the implemented schema; ve...
8,gold_dim_date,Shared,Shared calendar dimension used to align staffi...,One calendar date.,Internal Analytical,None


In [12]:
# Example 2: discover all Enforcement datasets across Medallion layers
pd.read_sql_query(
    '''
    SELECT
        layer,
        table_name,
        description,
        source_system,
        refresh_frequency
    FROM governance_data_catalog
    WHERE domain = 'Enforcement'
    ORDER BY
        CASE layer
            WHEN 'Bronze' THEN 1
            WHEN 'Silver' THEN 2
            WHEN 'Gold' THEN 3
            ELSE 4
        END,
        table_name
    ''',
    conn
)


,layer,table_name,description,source_system,refresh_frequency
0,Bronze,bronze_ice_enforcement_metrics,Structured enforcement metrics derived from IC...,ICE Annual Reports / metric extraction process,Prototype annual-report batch load
1,Bronze,bronze_ice_enforcement_pdfs,Page-level text extracted from ICE annual repo...,ICE Annual Reports,Prototype annual-report batch load
2,Silver,silver_enforcement,Standardized enforcement records linking extra...,bronze_ice_enforcement_pdfs + bronze_ice_enfor...,Prototype batch processing
3,Gold,gold_fact_enforcement,Curated enforcement metrics with source caveat...,silver_enforcement,Prototype batch processing


In [13]:
# Example 3: look up a business term
pd.read_sql_query(
    '''
    SELECT
        term,
        definition,
        domain,
        related_table,
        related_column,
        notes
    FROM governance_business_glossary
    WHERE term IN ('Accrual-to-Cash Gap', 'Restatement', 'Enforcement Metric')
    ORDER BY term
    ''',
    conn
)


,term,definition,domain,related_table,related_column,notes
0,Accrual-to-Cash Gap,A Gold analytical measure calculated as net op...,Funding,gold_fact_budget,accrual_to_cash_gap,This is a CAC prototype analytical measure.
1,Enforcement Metric,"A reported enforcement measure, such as an arr...",Enforcement,gold_fact_enforcement,metric_name,Metric definitions may vary by source/report; ...
2,Restatement,A later version of a Treasury financial statem...,Funding,gold_fact_treasury,is_restated,None


In [14]:
# Example 4: inspect documented columns for a Gold fact table
pd.read_sql_query(
    '''
    SELECT
        column_name,
        data_type,
        is_nullable,
        is_primary_key,
        business_definition,
        source_or_derivation
    FROM governance_data_dictionary
    WHERE table_name = 'gold_fact_treasury'
    ORDER BY is_primary_key DESC, column_name
    ''',
    conn
)


,column_name,data_type,is_nullable,is_primary_key,business_definition,source_or_derivation
0,treasury_key,TEXT,0,1,None,None
1,date_key,INTEGER,0,0,None,None
2,is_restated,INTEGER,0,0,Binary indicator identifying a restated Treasu...,Derived from restatement_flag during Gold proc...
3,position_billion_amount,REAL,1,0,None,None
4,record_fiscal_year,INTEGER,0,0,None,None
5,restatement_flag,TEXT,0,0,None,None
6,source_line_number,INTEGER,1,0,None,None
7,statement_fiscal_year,INTEGER,0,0,None,None
8,treasury_line_key,TEXT,0,0,None,None


## 8. Coverage summary

This final query shows how many datasets, columns, and glossary terms are documented.


In [15]:
coverage_df = pd.read_sql_query(
    '''
    SELECT 'Cataloged datasets' AS governance_artifact,
           COUNT(*) AS documented_items
    FROM governance_data_catalog

    UNION ALL

    SELECT 'Dictionary columns',
           COUNT(*)
    FROM governance_data_dictionary

    UNION ALL

    SELECT 'Business glossary terms',
           COUNT(*)
    FROM governance_business_glossary
    ''',
    conn
)

coverage_df


,governance_artifact,documented_items
0,Cataloged datasets,18
1,Dictionary columns,166
2,Business glossary terms,14


## 9. Close the database connection


In [16]:
conn.close()
print("Database connection closed.")


Database connection closed.


## What this notebook implements for Clarity Analytics Center governance

This prototype now demonstrates:

- **Metadata management** through dataset- and column-level metadata.
- **Data discoverability** through a queryable catalog.
- **Business terminology standardization** through a business glossary.
- **Ownership/stewardship documentation** through role-based accountability metadata.
- **Sensitivity documentation** at the dataset and column level.
- **Known-limitations documentation** so analysts can see important cautions before using data.
- **Governance metadata quality checks** so catalog, dictionary, and glossary references remain tied to the implemented database.

This is a lightweight prototype implementation. A production Clarity Analytics Center environment could later migrate these metadata assets into an enterprise catalog/governance platform and add workflow, approval, automated lineage, and role-based access controls.
